# Padel player detection – YOLO training (Colab)

**Standalone notebook** – not part of any app. Train a **person-only** YOLO model for padel. Output: **best.pt**.

Run this entirely in Google Colab. When done, download `best.pt` and use it in your app (e.g. CourtFlow: put in `models/best.pt` and use `--detection-model models/best.pt`).

**Before you start:**
- Enable **GPU**: Runtime → Change runtime type → GPU.
- Have your dataset ready: folder `padel_person/` with `data.yaml`, `images/train`, `images/val`, `labels/train`, `labels/val` (YOLO format, one class `person` = 0). Zip it as `padel_person.zip` if you will upload.

## 1. Install Ultralytics

In [ ]:
!pip install -q ultralytics

## 2. Get your dataset into Colab

**Choose one:** upload a zip, or use a path on Google Drive after mounting.

### Option A – Upload zip (run this cell and upload your `padel_person.zip`)

In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()  # pick padel_person.zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
# Assume zip contains folder 'padel_person'
dataset_path = os.path.abspath('padel_person')
if not os.path.isdir(dataset_path):
    # If zip had no top folder, use current dir
    dataset_path = os.path.abspath('.')
print('Dataset at:', dataset_path)

### Option B – Google Drive (mount Drive and set path to your dataset folder)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# dataset_path = '/content/drive/MyDrive/padel_person'  # change to your path
# print('Dataset at:', dataset_path)

## 3. Check dataset and data.yaml

In [ ]:
import os
import yaml

yaml_path = os.path.join(dataset_path, 'data.yaml')
if not os.path.isfile(yaml_path):
    raise FileNotFoundError(f'No data.yaml at {yaml_path}. Use: path, train, val, nc: 1, names: [person]')

# Set path to current dataset location (works after upload or Drive mount)
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = dataset_path
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

with open(yaml_path) as f:
    print(f.read())

for sub in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    p = os.path.join(dataset_path, sub)
    n = len([x for x in os.listdir(p)]) if os.path.isdir(p) else 0
    print(f'{sub}: {n} items')

## 4. Train

Uses YOLOv8 by default. Change `model=` to `yolo26n.pt` if your Ultralytics supports YOLO26. Output: `runs/detect/train/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

data_yaml = os.path.join(dataset_path, 'data.yaml')

model = YOLO('yolov8n.pt')  # or 'yolo26n.pt'
model.train(
    data=data_yaml,
    epochs=80,
    imgsz=640,
    batch=16,
    project='runs/detect',
    name='train',
)

## 5. Download best.pt

Run this to download the trained weights to your machine. Then copy `best.pt` into CourtFlow (e.g. `models/best.pt`) and run with `--detection-model models/best.pt`.

In [ ]:
from google.colab import files

best_pt = 'runs/detect/train/weights/best.pt'
if os.path.isfile(best_pt):
    files.download(best_pt)
else:
    print('best.pt not found. Check training completed and path:', best_pt)